# Ultralytics YOLOv8 Training Notebook

## This Section is for Training Yolo

#### insert Variable in the data section below

In [ ]:
import os
import shutil
import yaml
import json
import numpy as np
import glob
import optuna
import torch
from ultralytics import YOLO
import ultralytics.data.build as build
from ultralytics.data.dataset import YOLODataset

In [ ]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        super(YOLOWeightedDataset, self).__init__(*args, **kwargs)
        self.train_mode = "train" in getattr(self, "prefix", "")
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts
        self.class_weights = np.array(class_weights)
        self.agg_func = np.mean
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()
        print(f"✅ Using YOLOWeightedDataset with {len(self.counts)} classes.")
        print(f"Class counts: {self.counts.tolist()}")
        print(f"Sampling probabilities (first 10): {self.probabilities[:10]}")
    
    def count_instances(self):
        self.counts = [0 for _ in range(len(self.data.get("names", [])))]
        for label in getattr(self, "labels", []):
            cls = label.get('cls', np.array([])).reshape(-1).astype(int)
            for id in cls:
                if 0 <= id < len(self.counts):
                    self.counts[id] += 1
        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        weights = []
        for label in getattr(self, "labels", []):
            cls = label.get('cls', np.array([])).reshape(-1).astype(int)
            if cls.size == 0:
                weights.append(1.0)
                continue
            weight = float(self.agg_func(self.class_weights[cls]))
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        total_weight = float(sum(self.weights)) if len(self.weights) > 0 else 1.0
        if total_weight == 0:
            return [1.0 / len(self.weights)] * len(self.weights) if len(self.weights) > 0 else []
        return [w / total_weight for w in self.weights]

    def __getitem__(self, index):
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            if not self.probabilities:
                return self.transforms(self.get_image_and_label(index))
            idx = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(idx))

build.YOLODataset = YOLOWeightedDataset

In [ ]:
dataset = "../data/dataset_split/data.yaml"
model_path = "yolo11n-seg.pt"
study_name = "YOLO11_Marrows_2.1_tuning"
trial_group = "hyperparameters_tuning"
directory = f"optuna_study/{study_name}/{trial_group}"
storage_path = f"sqlite:///{study_name}.db"

os.makedirs(directory, exist_ok=True)

In [ ]:
def train_model(hyp, trial_num, use_default=False):
    trial_name = f"{trial_group}_{trial_num}"
    project_dir = os.path.join(directory, trial_name)

    if os.path.exists(project_dir):
        shutil.rmtree(project_dir)

    hyp_path = None
    if not use_default:
        hyp_path = f"{trial_name}_hyp.yaml"
        with open(hyp_path, 'w') as f:
            yaml.dump(hyp, f)

    device = "cpu"

    model = YOLO(model_path)

     # Train
    results = model.train(
        data=dataset,
        epochs=150,
        imgsz=640,
        batch=8,
        name=trial_name,
        cfg=hyp_path if hyp_path else None,
        patience=15,  # Early stopping
        device=device,
    )

    try:
        result_dir = project_dir
        best_weight = os.path.join(result_dir, "weights", "best.pt")
        if os.path.exists(best_weight):
            shutil.copy(best_weight, f"{trial_name}_best.pt")

        metrics_json = os.path.join(result_dir, "metrics.json")
        if os.path.exists(metrics_json):
            with open(metrics_json, 'r') as f:
                metrics_dict = json.load(f)
        else:
            metrics_dict = {}

        cm_file = os.path.join(result_dir, "confusion_matrix.png")
        if os.path.exists(cm_file):
            shutil.copy(cm_file, f"{trial_name}_confusion_matrix.png")

        csv_file = os.path.join(result_dir, "results.csv")
        if os.path.exists(csv_file):
            shutil.copy(csv_file, f"{trial_name}_results.csv")

        return metrics_dict.get("metrics/mAP50(B)", 0.0)

    except Exception as e:
        print(f"⚠️ Error saving results for trial {trial_num}: {e}")
        return 0.0

In [ ]:
def objective(trial):
    if trial.number == 0:
        print("🚀 Running baseline trial with default YOLOv11 hyperparameters...")
        return train_model(None, trial.number, use_default=True)
    
    hyp = {
    # Non-augmentasi
    "lr0": trial.suggest_float("lr0", 1e-5, 1e-1, log=True),
    "lrf": trial.suggest_float("lrf", 0.01, 1.0),
    "momentum": trial.suggest_float("momentum", 0.6, 0.98),
    "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.001),
    "warmup_epochs": trial.suggest_float("warmup_epochs", 0.0, 5.0),
    "warmup_momentum": trial.suggest_float("warmup_momentum", 0.0, 0.95),
    "box": trial.suggest_float("box", 0.02, 0.2),
    "cls": trial.suggest_float("cls", 0.2, 4.0),

    # Augmentasi aman untuk cell
    "hsv_h": trial.suggest_float("hsv_h", 0.0, 0.02),
    "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.15),
    "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.15),
    "degrees": trial.suggest_float("degrees", 0.0, 5.0),
    "translate": trial.suggest_float("translate", 0.0, 0.05),
    "scale": trial.suggest_float("scale", 0.9, 1.1),
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
}

    try:
        return train_model(hyp, trial.number)
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed: {e}")
        return 0.0

In [ ]:
if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        storage=storage_path,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=50)

    print("✅ Tuning complete!")
    print("Best Trial:")
    print(study.best_trial)

In [ ]:
best_trial = study.best_trial
best_trial_number = best_trial.number
best_model_path = f"runs/detect/trial_Hyper_{best_trial.number}/weights/best.pt"

In [ ]:
print(f"Best model saved at: {best_model_path}")

In [ ]:
print(study.trials)